# Review Notebook: Semantic Sentiment Analysis

**For studying, not presenting.** It explains everything in `Semantic_Presentation.ipynb`: what "semantic" means, how a transformer represents meaning, transfer learning and fine-tuning, how the training connects to the Week 1–4 lessons, and the final comparison with the rule-based approach.

| Part | What's inside |
|---|---|
| 1 | Results and the comparison in one place |
| 2 | "Semantic" = meaning as vectors (hands-on) |
| 3 | Transformers, tokenization, transfer learning |
| 4 | The training loop and its settings |
| 5 | Rule-based vs semantic |
| 6 | Lesson map (Weeks 1–4) |
| 7 | Background experiments |
| 8 | Questions the professor might ask |
| 9 | Self-check quiz |

Runs on Colab or locally (kernel **Python (sentiment_env)**). Part 2 needs `model/roberta_binary`, so run `Semantic_Presentation.ipynb` first. It also runs on a CPU.

In [1]:
# only runs on Google Colab: connect Google Drive and go to the project folder
import os, sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/Sentiment Analysis")   # change this if your folder has a different name

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

---
# Part 1: Results

| | Rule-based (Part 1) | Semantic (this notebook) |
|---|---|---|
| Method | VADER + ~1,040 scored words + phrases + 7 patterns + star/question rules | Twitter-RoBERTa (pretrained on sentiment), fine-tuned 1 epoch on 13,314 reviews |
| Validation accuracy | 89.4% | 93.2% |
| **Test accuracy** | **88.6%** | **94.6%** |

On the 2,854 test reviews: both right 2,486 · **semantic fixed 214** rule-based mistakes · rules right on **44** the semantic model missed · both wrong 110 (often reviews whose stars don't match their text).

**One-sentence summary:** *The semantic model is about 6 points more accurate because it understands a whole sentence in context. The rule-based system is faster, needs no GPU and can explain every decision.*

---
# Part 2: "Semantic" = meaning as vectors

The model turns every sentence into a vector of 768 numbers. Sentences with similar **meaning** get similar vectors, even if they share no words. We measure closeness with **cosine similarity** (1 = same direction).

In [2]:
tok = AutoTokenizer.from_pretrained("model/roberta_binary")
sem = AutoModelForSequenceClassification.from_pretrained("model/roberta_binary", output_hidden_states=True).eval()

def sentence_vector(text):
    with torch.no_grad():
        out = sem(**tok(text, return_tensors="pt"))
    return out.hidden_states[-1][0].mean(dim=0)      # average of the last layer's token vectors

cos = lambda a, b: torch.nn.functional.cosine_similarity(a, b, dim=0).item()

anchor = "The service was terrible"
for other in ["The staff was really rude to us", "Nobody helped us for 20 minutes",
              "The service was excellent", "The service was not terrible at all"]:
    print(f"{cos(sentence_vector(anchor), sentence_vector(other)):.3f}   '{anchor}'  vs  '{other}'")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

0.883   'The service was terrible'  vs  'The staff was really rude to us'
0.891   'The service was terrible'  vs  'Nobody helped us for 20 minutes'
0.155   'The service was terrible'  vs  'The service was excellent'
0.297   'The service was terrible'  vs  'The service was not terrible at all'


"Nobody helped us for 20 minutes" shares almost no words with "The service was terrible" but lands close to it. "The service was excellent" shares four of five words but is far away. **Rules compare words; the semantic model compares meanings.**

In [3]:
def word_vector(sentence, word):
    enc = tok(sentence, return_tensors="pt")
    with torch.no_grad():
        hidden = sem(**enc).hidden_states[-1][0]
    tokens = tok.convert_ids_to_tokens(enc["input_ids"][0])
    return hidden[[k for k, t in enumerate(tokens) if t.strip("Ġ").lower() == word][0]]

a = word_vector("The burger was really good", "good")
b = word_vector("The burger was not good", "good")
c = word_vector("The fries were really good", "good")
print(f"'good' in 'really good' (burger) vs 'not good':           {cos(a, b):.3f}")
print(f"'good' in 'really good' (burger) vs 'really good' (fries): {cos(a, c):.3f}")

'good' in 'really good' (burger) vs 'not good':           0.463
'good' in 'really good' (burger) vs 'really good' (fries): 0.986


The **same word** gets a **different vector depending on its context**. In VADER, "good" is always 1.9, and a separate negation rule has to flip it. In the transformer, context is built into the representation itself. That's what **contextual / semantic** means.

In [4]:
def p_positive(text):
    with torch.no_grad():
        return torch.softmax(sem(**tok(text, return_tensors="pt")).logits, dim=1)[0, 1].item()

for t in ["I asked for no pickles and got extra pickles", "Pay your morning manager more! She is a rockstar",
          "They were fast, but my fries were cold", "Oh great, cold fries again. Just what I wanted"]:
    print(f"P(positive) = {p_positive(t):.2f}   {t}")

P(positive) = 0.11   I asked for no pickles and got extra pickles
P(positive) = 0.99   Pay your morning manager more! She is a rockstar
P(positive) = 0.93   They were fast, but my fries were cold
P(positive) = 0.59   Oh great, cold fries again. Just what I wanted


The first two have no typical sentiment words, so the rules fail on them, and the semantic model gets both right.

The other two show its limits. "They were fast, but my fries were cold" comes out **positive (0.93)**, though most people would call it a complaint. Mixed reviews are hard because both readings are reasonable. Here the rule-based system actually does better: VADER's "but" rule gives the part after "but" more weight, so it scores the review negative (-0.27). The sarcastic one gets 0.59, so the model is unsure. Being more accurate on average doesn't mean being right on every review.

---
# Part 3: Transformers, tokenization, transfer learning

## 3.1 How a transformer reads a review

1. **Tokenize:** split into word pieces ("disapointed" → `dis` `ap` `ointed`), so no word is ever "unknown".
2. **Embed:** each token becomes a vector.
3. **12 layers of attention:** in each layer every token looks at every other token and mixes in information from the ones that matter ("good" pays attention to "not").
4. **Classifier head:** a small linear layer turns the sentence representation into 2 numbers, and **softmax** turns them into P(negative), P(positive).

About **125 million** weights. Inside it's still Weeks 1 and 4: layers of weighted sums + activations, trained by backpropagation.

In [5]:
for text in ["The fries were great", "disapointed", "McFlurry machine broken AGAIN"]:
    print(f"{text!r:32s} -> {tok.tokenize(text)}")
print("\nparameters:", f"{sum(p.numel() for p in sem.parameters()):,}")

'The fries were great'           -> ['The', 'Ġfries', 'Ġwere', 'Ġgreat']
'disapointed'                    -> ['dis', 'ap', 'ointed']
'McFlurry machine broken AGAIN'  -> ['Mc', 'Fl', 'urry', 'Ġmachine', 'Ġbroken', 'ĠAGA', 'IN']

parameters: 124,647,170


## 3.2 Transfer learning and fine-tuning

| | From scratch | Fine-tuning (ours) |
|---|---|---|
| Starting weights | random | already trained for sentiment on ~124M tweets |
| Data needed | millions | thousands (13,314) |
| Learning rate | ~0.001 | **0.00002** |
| Epochs | many | **1** |
| Time | days | ~75 s on our RTX 3060 (~2–3 min on a Colab T4) |

- **Why so small a learning rate?** Big steps would overwrite what the model already knows (Week 4: "too high → divergence", except here what gets destroyed is the pretrained knowledge).
- **Why 1 epoch?** Validation accuracy was best after epoch 1 and slightly lower after, the start of **overfitting** (Week 1 s25, Week 4 s33). The model already knew sentiment.
- **Why a Twitter model?** Tweets are short, informal and full of typos, like Google reviews. Of three pretrained models we tried, it scored best (93.2% vs 92.7% and 92.1%).

---
# Part 4: The training loop

```python
optimizer.zero_grad()
loss = cross_entropy(model(batch), labels)   # forward pass + loss (W1 s17-18, W4 s23, s29)
loss.backward()                              # backpropagation (W1 s20, W4 s31)
optimizer.step()                             # AdamW update (W1 s19: "Gradient Descent, Adam")
scheduler.step()                             # warmup + linear decay of the learning rate
```

| Setting | What and why |
|---|---|
| **AdamW**, lr 2e-5 | Adam + weight decay (a regularizer, one of the overfitting fixes in W1 s25) |
| **Warmup** (first 6% of steps) | The new classifier head starts random, so its first gradients are messy. Small steps at the start protect the pretrained weights |
| **Linear decay** | Smaller steps toward the end, to settle instead of bouncing |
| **Batch size 32**, max 128 tokens | 13,314 / 32 = 417 weight updates in one epoch. 95% of reviews are well under 128 tokens |
| **Mixed precision** (`autocast`) | 16-bit math on the GPU for speed. Purely a speed trick |
| **Cross-entropy + softmax**, 2 outputs | Classification loss from W1 s18. 2 softmax outputs are equivalent to 1 sigmoid output (W4 s20 "binary → sigmoid") |
| **`model.eval()` + `torch.no_grad()`** for predicting | Inference = forward pass only (W4 s26–27). `eval()` turns off the model's built-in dropout |
| **GPU** | ~75 s vs 30–60 min on a CPU, because transformers are huge matrix multiplications (W1 s22, W3 s15) |

---
# Part 5: Rule-based vs semantic

| | Rule-based | Semantic |
|---|---|---|
| Knowledge comes from | people + word statistics, written as lists | learned from labeled examples |
| Understands context | only through fixed rules (negation, "but", phrases) | yes, built into the vectors |
| Unknown / misspelled words | score 0 | split into known pieces |
| Explainable | fully (every point traceable) | hard ("black box") |
| Needs GPU / labeled data | no / only to build the lists | yes / yes |
| Size / speed | a few hundred KB, instant | ~500 MB, slower |
| Test accuracy | 88.6% | 94.6% |

**When to choose which:** rules when there's no labeled data or GPU, when every decision must be justified, or for a quick baseline. Semantic when accuracy matters most and you have labeled data and compute.

---
# Part 6: Lesson map (Weeks 1–4)

| Lesson topic | Where it appears |
|---|---|
| W1 s3–4: DL learns features automatically | The transformer learns its own features (vectors). The rules are hand-made features |
| W1 s5, W4 s6: neuron `f(w·x + b)` | Every layer of RoBERTa and its classifier head |
| W1 s9–15, W4 s15–20: activations, softmax | Softmax on the 2 outputs. GELU (a smooth ReLU) inside the layers |
| W1 s16, W4 s13: deep networks / MLP | 12 transformer layers, each containing an MLP block |
| W1 s17, W4 s23–27: forward pass, training vs inference | Training loop vs `semantic_proba()` with `eval()` and `no_grad()` |
| W1 s18–19, W4 s29: cross-entropy loss | `cross_entropy(logits, labels)` |
| W1 s19: optimization, **Adam** | **AdamW** |
| W1 s20, W4 s30–31: backprop, `w_new = w_old − α·∇L` | `loss.backward()` + `optimizer.step()` |
| W4 s32: learning rate | 2e-5 with warmup + decay |
| W1 s25, W4 s33–35: overfitting, dropout | 1 epoch. Dropout (0.1) built into RoBERTa |
| W1 s22, W2 s10–13, W3 s8–15: GPU, CUDA, tensors | `device = "cuda"`, tensors on GPU, mixed precision |
| W2 s14–16, W3 s4–13: venv, requirements, kernels | `sentiment_env`, `requirements.txt`, SETUP.md |
| W1 s24: supervised learning | Star ratings are the labels |
| W1 s26: NLP applications | The whole project |

---
# Part 7: Background experiments (good to know for Q&A)

| Experiment | Result | Lesson |
|---|---|---|
| 3 classes incl. neutral (TF-IDF + our Week 4 MLP) | 74.3% test, neutral F1 only 0.38 | 3★ reviews are ambiguous. Even big pretrained models got only 74–77% |
| Word-count features (TF-IDF) + Naive Bayes / Logistic Regression / SVM / NB-SVM / MLP, all tuned | all ~90% | Word counting has a ceiling. Tuning can't add missing information |
| Fine-tuning 3 pretrained transformers | 92.1% / 92.7% / **93.2%** validation | A sentiment-pretrained start is best |
| Ensemble of MLP + RoBERTa | about the same as RoBERTa alone | Not worth the extra complexity |

---
# Part 8: Questions the professor might ask

<details><summary><b>Q1. What does "semantic" mean in your project?</b></summary>

Based on meaning. The model represents each sentence, and each word in its context, as a vector, so similar meanings get similar vectors. It decides sentiment from the meaning of the whole review instead of looking up individual words.
</details>

<details><summary><b>Q2. What model did you use?</b></summary>

`cardiffnlp/twitter-roberta-base-sentiment-latest`, a RoBERTa transformer pretrained for sentiment on about 124 million tweets, fine-tuned on our 13,314 training reviews for one epoch.
</details>

<details><summary><b>Q3. What's fine-tuning / transfer learning?</b></summary>

Starting from a model already trained on a big dataset and continuing training on our data with a small learning rate. It keeps its general language knowledge and adapts to McDonald's reviews.
</details>

<details><summary><b>Q4. How is this related to what we learned?</b></summary>

Same training loop: forward pass, cross-entropy loss, backpropagation, an Adam-type optimizer (AdamW), softmax output, and overfitting control. The difference is size (12 layers, ~125M weights) and attention layers that let each word use the others as context.
</details>

<details><summary><b>Q5. Why only 1 epoch? Why such a small learning rate?</b></summary>

Validation accuracy peaked after epoch 1 and dropped slightly after, so more epochs would overfit. The small learning rate (2e-5) adjusts the pretrained knowledge without destroying it.
</details>

<details><summary><b>Q6. Why this pretrained model?</b></summary>

We compared three on validation: DistilRoBERTa 92.1%, RoBERTa-base 92.7%, Twitter-RoBERTa-sentiment 93.2%. It already knew sentiment and short informal text.
</details>

<details><summary><b>Q7. Results compared to rule-based?</b></summary>

Semantic 94.6% vs rule-based 88.6% on the same 2,854 test reviews. The semantic model fixed 214 rule-based mistakes, and the rules got 44 right that it missed.
</details>

<details><summary><b>Q8. Where does the semantic model still fail?</b></summary>

Sarcasm, genuinely mixed reviews, and reviews whose star rating contradicts the text, which are label problems no model can fix.
</details>

<details><summary><b>Q9. Why is the rule-based approach still useful?</b></summary>

It needs no GPU or training, runs instantly anywhere, and every decision is explainable. Useful when there's no labeled data or when you must justify each result.
</details>

<details><summary><b>Q10. Did you need a GPU?</b></summary>

For training, practically yes: ~75 s on our RTX 3060 vs 30–60 min on a CPU. Predicting works on a CPU, and the notebook can load our saved model (`TRAIN_TRANSFORMER = False`).
</details>

<details><summary><b>Q11. How do you know the test result is honest?</b></summary>

Every choice (pretrained model, epochs, learning rate) was made on the validation set. The test set was used once. Both approaches use exactly the same split, and duplicates were removed so no review appears in both train and test.
</details>

<details><summary><b>Q12. Why AdamW and not plain Adam?</b></summary>

AdamW adds weight decay (applied separately from the gradient step), which keeps weights small and reduces overfitting. It's the standard optimizer for transformers.
</details>

<details><summary><b>Q13. Why did you drop 3-star reviews?</b></summary>

They're usually mixed, so there's no reliable right answer. In our 3-class test even large pretrained models got only 74–77%. Public datasets built from star ratings drop the middle rating too.
</details>

<details><summary><b>Q14. Could the semantic model reach 100%?</b></summary>

No. Some labels contradict the text ("Took 20 minutes to get my order" with 4–5 stars). Of the 2,854 test reviews, 110 are wrong for both approaches, many of them for this reason.
</details>

---
# Part 9: Self-check quiz

<details><summary><b>1. What does "contextual" mean for word vectors?</b></summary>The same word gets a different vector depending on the words around it.</details>

<details><summary><b>2. What does a tokenizer do with a misspelled word?</b></summary>Splits it into smaller known pieces instead of dropping it.</details>

<details><summary><b>3. Define transfer learning.</b></summary>Reusing a model trained on a large task as the starting point for a new task.</details>

<details><summary><b>4. The 4 steps of a training step?</b></summary>Forward pass, loss, backpropagation, optimizer update.</details>

<details><summary><b>5. Why warmup?</b></summary>The new classifier head starts random, and small early steps protect the pretrained weights from its messy first gradients.</details>

<details><summary><b>6. Which activation gives the final probabilities?</b></summary>Softmax over the 2 outputs.</details>

<details><summary><b>7. What does model.eval() change?</b></summary>It turns off dropout for predictions.</details>

<details><summary><b>8. 13,314 reviews, batch size 32. Weight updates in one epoch?</b></summary>417.</details>

<details><summary><b>9. Give one advantage of rule-based over semantic.</b></summary>Explainable, instant, no GPU or training needed.</details>

<details><summary><b>10. Final test accuracies?</b></summary>Rule-based 88.6%, semantic 94.6%.</details>